# Monthly Sales Time-Series Exploration

## Business purpose
Explore monthly sales trends, annual performance, and recurring month-of-year patterns using complete months only. This is descriptive analysis, not proof of causal drivers.

The analytical period is January 2013 through July 2017. The partial month of August 2017 is excluded.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'src').is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate project root.')

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
monthly_sales_complete = pd.read_csv(PROCESSED_DIR / 'monthly_sales_complete.csv')
print('Project root:', PROJECT_ROOT)
print('Loaded complete monthly sales rows:', len(monthly_sales_complete))

## Phase 3.8 — Monthly Sales Time-Series Exploration

In [ ]:
# Step 3.8.1 — Prepare complete monthly Sales time series

monthly_ts = monthly_sales_complete.copy()

# Convert month from YYYY-MM text/Period to datetime month-start
monthly_ts["month"] = pd.to_datetime(
    monthly_ts["month"].astype(str)
)

# Sort chronologically
monthly_ts = monthly_ts.sort_values("month").reset_index(drop=True)

print("Rows:", len(monthly_ts))
print("Start:", monthly_ts["month"].min())
print("End:", monthly_ts["month"].max())

display(
    monthly_ts[
        ["month", "total_sales"]
    ].head()
)

display(
    monthly_ts[
        ["month", "total_sales"]
    ].tail()
)

In [ ]:
# Step 3.8.2 — Validate monthly time-series continuity

expected_ts_months = pd.date_range(
    start=monthly_ts["month"].min(),
    end=monthly_ts["month"].max(),
    freq="MS"
)

actual_ts_months = monthly_ts["month"]

missing_ts_months = expected_ts_months.difference(
    actual_ts_months
)

duplicate_ts_months = monthly_ts["month"].duplicated().sum()

ts_continuity_qa = {
    "expected_months": len(expected_ts_months),
    "actual_months": len(actual_ts_months),
    "missing_months": missing_ts_months.tolist(),
    "duplicate_months": int(duplicate_ts_months),
    "chronological_order": actual_ts_months.is_monotonic_increasing,
    "continuity_pass": (
        len(expected_ts_months) == len(actual_ts_months)
        and len(missing_ts_months) == 0
        and duplicate_ts_months == 0
        and actual_ts_months.is_monotonic_increasing
    )
}

ts_continuity_qa

In [ ]:
# Step 3.8.3 — Monthly Sales movement

monthly_ts["sales_change"] = (
    monthly_ts["total_sales"]
    .diff()
)

monthly_ts["sales_growth_pct"] = (
    monthly_ts["total_sales"]
    .pct_change() * 100
)

display(
    monthly_ts[
        [
            "month",
            "total_sales",
            "sales_change",
            "sales_growth_pct"
        ]
    ].head(12)
)

### Step 3.8.4 — Create the monthly Sales trend visualization

In [ ]:
# Step 3.8.4 — Monthly Sales Trend

import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(
    monthly_ts["month"],
    monthly_ts["total_sales"]
)

plt.title("Monthly Total Sales — Complete Months")
plt.xlabel("Month")
plt.ylabel("Total Sales")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Step 3.8.5 — Annual Sales Summary

monthly_ts["year"] = monthly_ts["month"].dt.year

annual_sales = (
    monthly_ts
    .groupby("year")
    .agg(
        annual_sales=("total_sales", "sum"),
        months_available=("month", "nunique")
    )
    .reset_index()
)

annual_sales

In [ ]:
# Step 3.8.6 — Annual Sales Growth

# Calculate growth only when both the current and previous years
# contain all 12 months. This avoids comparing a partial year with
# a complete year.

annual_sales["is_complete_year"] = (
    annual_sales["months_available"] == 12
)

previous_year_complete = annual_sales["is_complete_year"].shift(1).fillna(False)

annual_sales["annual_growth_pct_valid"] = (
    annual_sales["annual_sales"]
    .pct_change()
    .where(
        annual_sales["is_complete_year"] & previous_year_complete
    )
    * 100
)

annual_sales


In [ ]:
# Step 3.8.6 — Validate complete calendar-year comparisons

annual_sales


In [ ]:
# Show only valid full-year comparisons

display(
    annual_sales[
        annual_sales["is_complete_year"]
    ][
        [
            "year",
            "annual_sales",
            "months_available",
            "annual_growth_pct_valid"
        ]
    ]
)

In [ ]:
# Step 3.8.7 — Month-of-year Sales analysis

monthly_ts["month_number"] = (
    monthly_ts["month"].dt.month
)

monthly_ts["month_name"] = (
    monthly_ts["month"].dt.strftime("%b")
)

seasonality_summary = (
    monthly_ts
    .groupby(
        ["month_number", "month_name"]
    )
    .agg(
        average_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        minimum_sales=("total_sales", "min"),
        maximum_sales=("total_sales", "max"),
        observations=("total_sales", "count")
    )
    .reset_index()
    .sort_values("month_number")
)

seasonality_summary

In [ ]:
# Step 3.8.8 — Relative month-of-year Sales index

overall_average_sales = monthly_ts["total_sales"].mean()

seasonality_summary["seasonal_index"] = (
    seasonality_summary["average_sales"]
    / overall_average_sales
)

seasonality_summary

## Step 3.8.9 — Normalize Sales Within Each Year

### 1. Calculate each year's average monthly Sales

In [ ]:
year_average_sales = (
    monthly_ts
    .groupby("year")["total_sales"]
    .transform("mean")
)

monthly_ts["year_average_sales"] = year_average_sales

### 2. Calculate each month's Sales relative to its year's average

In [ ]:
monthly_ts["normalized_sales_index"] = (
    monthly_ts["total_sales"]
    / monthly_ts["year_average_sales"]
)

### 3. Build the normalized seasonal profile

In [ ]:
normalized_seasonality = (
    monthly_ts
    .groupby(["month_number", "month_name"])
    .agg(
        average_normalized_index=("normalized_sales_index", "mean"),
        median_normalized_index=("normalized_sales_index", "median"),
        observations=("normalized_sales_index", "count")
    )
    .reset_index()
    .sort_values("month_number")
)

normalized_seasonality

### Step 3.8.10 — Visualize the Normalized Seasonal Profile

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    normalized_seasonality["month_name"],
    normalized_seasonality["average_normalized_index"],
    marker="o"
)

plt.axhline(
    1.0,
    linestyle="--"
)

plt.title("Normalized Monthly Sales Seasonality")
plt.xlabel("Month")
plt.ylabel("Normalized Sales Index")
plt.tight_layout()
plt.show()

## Step 3.8.11 — Check whether the seasonal pattern repeats across years
### 1. Create a year-by-month seasonal index table

In [ ]:
seasonality_by_year = monthly_ts.pivot(
    index="year",
    columns="month_name",
    values="normalized_sales_index"
)

month_order = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

seasonality_by_year = seasonality_by_year[month_order]

seasonality_by_year.round(3)

### 2. Check consistency of the seasonal pattern

In [ ]:
seasonality_consistency = (
    monthly_ts
    .groupby(["month_number", "month_name"])
    .agg(
        above_average_years=(
            "normalized_sales_index",
            lambda x: (x > 1).sum()
        ),
        below_average_years=(
            "normalized_sales_index",
            lambda x: (x < 1).sum()
        ),
        observations=("normalized_sales_index", "count")
    )
    .reset_index()
    .sort_values("month_number")
)

seasonality_consistency

### 3. Visualize the year-by-month pattern

In [ ]:
plt.figure(figsize=(14, 6))

for year in seasonality_by_year.index:
    plt.plot(
        month_order,
        seasonality_by_year.loc[year],
        marker="o",
        label=str(year)
    )

plt.axhline(1.0, linestyle="--")

plt.title("Monthly Sales Seasonality by Year")
plt.xlabel("Month")
plt.ylabel("Normalized Sales Index")
plt.legend(title="Year")
plt.tight_layout()
plt.show()